# Computation of stringy mixed Hodge polynomials for character varieties of abelian groups

Let $G$ be a connected complex reductive group. Choose a maximal torus $T \subseteq G$ and let $W$ be the associated Weyl group. This script computes the stringy invariants of the variety
$$
    \hat{\mathfrak{X}}_G^r = T^r / W,
$$
for all $r \geq 1$, with respect to the diagonal action of $W$ on $T^r$. The variety $\hat{\mathfrak{X}}_G^r$ is the normalization of the identity component of the $G$-character variety of representations of the abelian group $\mathbb{Z}^r$,

$$
    \mathfrak{X}_G(\mathbb{Z}^r) = \text{Hom}(\mathbb{Z}^r, G) \,//\, G.
$$

Concretely, this script computes the stringy mixed Hodge polynomials of $T^r/W$, given by

$$
    \mu^{\text{str}}(T^r/W)(t, u, v) = \sum_{[w] \in \text{Conj}(W)} \mu((T^r)^w / C(w))\, (t^{2}uv)^{\text{age}(w)},
$$
where
$$
    \text{age}(w) = \frac{r}{2} \text{codim}_{T}(T^w).
$$
Here, $\text{Conj}(W)$ denotes the set of conjugacy classes of $W$, $C(w)$ is the centralizer of $w \in w$, and $(T^r)^w$ denotes the fixed-point locus of $w$ on $T^r$.

In order to do so, we implement the formula established in Theorem A of [FGPZ], which proves that
$$
        \mu^{\text{str}}(T^r/W)(t,u,v)=\sum_{[w] \in \text{Conj}(W)}
        \frac{(t^{2}uv)^{\frac{r}{2}\text{rk}\, \text{Im}(I_\Lambda -w)}}{|C(w)|}\sum_{g \in C(w)} \, n_{w}(g)^r \det\left(I_{\Phi_w} + tuv\,\phi_w(g)\right)^r.
$$
In this formula, we consider:
- $\Lambda = X^*(T)$ is the character lattice of $T$.
- For $w \in W$, $\Phi_w$ is the free part of
$$
    \text{coker}(I_\Lambda - w) = \Lambda \,/\, \text{Im}(I_\Lambda - w).
$$
- For $g \in C(w)$, $\phi_w(g): \Phi_w \to \Phi_w$ is the map induced by $g$ on the free part $\Phi_w$ of $\text{coker}(I_\Lambda - w)$.
- For $g \in C(w)$,
$$
    n_{w}(g) = |D_w\,/\,\text{Im}(I_{D_w} - \tau_w(g))|,
$$
where $D_w$ is the torsion part of $\text{coker}(I_\Lambda - w)$ and $\tau_w(g): D_w \to D_w$ is the map induced by $g$ on $D_w$.

### Reference

- [FGPZ] Carlos Florentino, Ángel González-Prieto and Alfonso Zamora, *Stringy invariants for abelian character varieties*, 2026.

In [1]:
var('q, r, d, t, u, v')

(q, r, d, t, u, v)

In [2]:
def make_lattice_action(L, W):
    """
    Returns rho(w), the integral matrix of w acting on the Z-basis of L.

    L must be a W-stable submodule of the domain of W.

    Input:
        L : Lattice
        W : Weyl group

    Output: Function rho that sends an element of W into its matrix representation on L.
    """
    L_basis = list(L.basis())
    cache = {}

    def rho(w):
        # Representatives of centralizer conjugacy classes may have
        # the centralizer as parent, so coerce them back into W.
        try:
            w = W(w)
        except (TypeError, ValueError):
            w = W(w.matrix())

        if w in cache:
            return cache[w]

        columns = []

        for b in L_basis:
            # Lift b from L to the ambient weight lattice P
            b_ambient = L.lift(b)

            # Apply w in P
            image_ambient = w.action(b_ambient)

            # Pull the result back to L
            try:
                image_L = L.retract(image_ambient)
            except ValueError:
                raise ValueError(
                    "The proposed lattice is not W-stable."
                )

            columns.append(vector(ZZ, image_L.to_vector()))

        # Columns are the coordinates of the images of the basis vectors
        M = matrix(ZZ, columns).transpose()

        cache[w] = M
        return M

    return rho
    
def compute_coker(M):
    """
    Computes the algebraic information of the cokernel of the endomorphism I-M.

    Applies the Smith Normal form and returns the decomposition as well as the indices
    corresponding to the torsion and free parts.

    Input:
        M: Endomorphism

    Output:
        S, U, V : Smith normal form S = U*(I-M)*V
        d : Invariant factors of coker(I-M)
        torsion_indices : Indices of the SNF corresponding to the torsion part
        free_indices : Indices of the SNF corresponding to the free part
        rank : Rank of the image of I-M
    """
    n = M.nrows()

    A = identity_matrix(ZZ, n) - M

    # Smith normal form.
    # It returns S, U, V with S = U*A*V.
    S, U, V = A.smith_form(transformation=True)

    # Diagonal entries of the Smith form
    d = [ZZ(S[i, i]) for i in range(n)]

    # Ignore trivial Z/1 factors
    torsion_indices = [i for i, di in enumerate(d) if abs(di) not in [0, 1]]
    free_indices = [i for i, di in enumerate(d) if di == 0]

    return S, U, V, d, torsion_indices, free_indices, A.rank()

def induced_on_coker(U, torsion_indices, free_indices,  G):
    """
    Given the cokernel group given by U with free part given by free_indices and
    torsion part given by torsion_indices with invariant factors d, computes
    the induced map of G on the cokernel.

    Input:
        U : Generators of the torsion and free part of the module
        torsion_indices : Indices of the torsion part
        free_indices : Indices of the free part
        G : Endomorphism to descend.

    Output:
        free_part : Free part of G.
        torsion_part : Torsion part of G.
    """
    #G = matrix(ZZ, g.matrix())
    B = U * G * U.inverse()

    # Free part Z^r
    B_free = B.matrix_from_rows_and_columns(free_indices, free_indices)

    # Torsion part direct sum Z/d_i
    B_torsion = B.matrix_from_rows_and_columns(torsion_indices, torsion_indices)
    
    return B_free, B_torsion

In [3]:
def quotient_torsion_by_I_minus_B(torsion_moduli, B_torsion):
    """
    Computes F / Im(I - B_torsion), where

        F = direct sum_i Z/d_i Z

    and B_torsion is the matrix of an endomorphism of F.

    Input:
        torsion_moduli : invariant factors of F
        B_torsion      : integer matrix, with row i read modulo d_i

    Output:
        invariant factors of F / Im(I - B_torsion)
    """

    k = len(torsion_moduli)

    if k == 0:
        return []

    D = diagonal_matrix(ZZ, torsion_moduli)
    I = identity_matrix(ZZ, k)

    M = I - matrix(ZZ, B_torsion)

    # Columns of D impose d_i e_i = 0.
    # Columns of M impose (I - B_torsion)(e_j) = 0 in the quotient.
    P = D.augment(M)

    ed = P.elementary_divisors()

    # Keep only nontrivial finite cyclic factors
    invariants = [abs(ZZ(a)) for a in ed if abs(ZZ(a)) not in [0, 1]]

    return invariants

In [4]:
def compute_stringy_MHpolynomial(L, W, r0 = 2, compact_support = False, Higgs = False, verbose = True):
    """
    Computes the stringy mixed Hodge polynomial of T^r / W, where
    
        T = Hom(L, C^*)
    
    is the algebraic torus associated with the W-lattice L, W acts
    diagonally on T^r, and r = r0.
    
    Input:
        L               : lattice equipped with an action of W
        W               : Weyl group acting on L
        r0              : number of copies of T
        compact_support : if True, computes the compactly supported
                          stringy mixed Hodge polynomial
        Higgs           : if True, instead of computing the stringy mixed
                          Hodge polynomial of T^r / W, computes the one of
                          the moduli space of G-Higgs bundles on an abelian
                          variety of dimension r/2 (so r must be even).
        verbose         : if True, prints the progress and the contribution
                          of each conjugacy class
    
    Output:
        stringy mixed Hodge polynomial of T^r / W, expressed in the
        variables t and q = u*v
    """
    
    classes = list(W.conjugacy_classes())
    
    MH_poly = 0
    rho = make_lattice_action(L, W) # Computes the matrices of the action of W on L

    counter = 0 # Counter to keep track of the progress
    
    for c in classes: # First sum on conjugacy classes of elements of W
        counter += 1

        if verbose:
            print('CONJUGACY CLASS ' + str(counter) + '/' + str(len(classes)))
        
        w = W(c.representative()) # Representative w of the conjugacy class c
        Mw = rho(w)
        Cw = W.centralizer(w)

        Cw_classes = Cw.conjugacy_classes()
        if verbose:
            print('w = ')
            print(w)
            print('Order of centralizer: ' + str(Cw.order()))
            print('Number of conjugacy classes: ' + str(len(Cw_classes)))

        # Compute coker(I-Mw)
        S, U, V, d, torsion_indices, free_indices, rank_image = compute_coker(Mw)
        
        i = 0
        MH_sector = 0
        torsion_moduli = [abs(d[i]) for i in torsion_indices]
        free_rank = len(free_indices)
        
        for g_class in Cw_classes: # Second sum on conjugacy classes of elements of C(w)
            if verbose:
                i += 1
                if i % 10 == 0:
                    print('\t Computed ' + str(i) + '/' + str(len(Cw_classes)))

            # Coerce from the centralizer back into W
            try:
                g = W(g_class.representative())
            except (TypeError, ValueError):
                g = W(g_class.representative().matrix())

            # Matrix of g on the same character lattice L
            Mg = rho(g)
            # This must hold because g belongs to C_W(w): Mg * Mw = Mw * Mg

            B_free, B_torsion = induced_on_coker(U, torsion_indices, free_indices, Mg)
            invariants = quotient_torsion_by_I_minus_B(torsion_moduli, B_torsion)

            if compact_support:
                deter = det(t*q*identity_matrix(free_rank) + B_free)
            else:
                deter = det(identity_matrix(free_rank) + t*q*B_free)

            if not Higgs:
                MH_sector += g_class.cardinality()*(prod(invariants)*deter)^r0
            else:
                MH_sector += g_class.cardinality()*(prod(invariants)^r0*deter.subs(q == u)^(r0/2)*deter.subs(q == v)^(r0/2))
                
        age = (r0/2)*rank_image

        if not Higgs:
            if compact_support:
                contribution = 1/Cw.order()*MH_sector*q^age*t^(r0*L.rank())
            else:
                contribution = 1/Cw.order()*MH_sector*(t^2*q)^age

        else:
            if compact_support:
                contribution = 1/Cw.order()*MH_sector*t^(r0*L.rank())*(u*v)^(r0/2*L.rank())
            else:
                contribution = 1/Cw.order()*MH_sector*(t^2*u*v)^age        
        
        if verbose:
            print('CONTRIBUTION: ' + str(contribution) + '\n----------------\n')
        
        MH_poly += contribution
    
    return MH_poly.simplify_full()

## Example of use for $G = \text{SL}_3$

$\text{SL}_3$ has root system of type $A_2$ and its character lattice is the root lattice, $\Lambda = Q$.

In [5]:
R = RootSystem("A2") # Define the root system

Q = R.root_lattice()             # Root lattice
P = R.weight_lattice()           # Weight lattice

L = P.submodule(Q.basis())       # The character lattice is the sublattice of P corresponding to Q

W = P.weyl_group()               # Associated Weyl group

In [6]:
# Character variety

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)

print('Character variety')
show(MH_poly)
show(MH_poly.subs(r==2).simplify_full())

CONJUGACY CLASS 1/3
w = 
[1 0]
[0 1]
Order of centralizer: 6
Number of conjugacy classes: 3
CONTRIBUTION: 1/3*(3*q^2*t^2 - (2*q*t - 1)*(q*t + 1))^r + 1/6*((q*t + 1)^2)^r + 1/2*(-(q*t + 1)*(q*t - 1))^r
----------------

CONJUGACY CLASS 2/3
w = 
[-1  0]
[ 1  1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (q*t^2)^(1/2*r)*(q*t + 1)^r
----------------

CONJUGACY CLASS 3/3
w = 
[-1 -1]
[ 1  0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: 3^r*(q*t^2)^r
----------------

Character variety


3^r*(q*t^2)^r + (q*t^2)^(1/2*r)*(q*t + 1)^r + 1/6*(q^2*t^2 + 2*q*t + 1)^r + 1/3*(q^2*t^2 - q*t + 1)^r + 1/2*(-q^2*t^2 + 1)^r

2*q^2*t^3 + (q^4 + q^3 + 9*q^2)*t^4 + (q^2 + q)*t^2 + 1

In [7]:
# Moduli space of Higgs bundles

MH_poly_Higgs = compute_stringy_MHpolynomial(L, W,  r0 = 2*d, compact_support = False, Higgs = True)

print('Moduli space of Higgs bundles')
show(MH_poly_Higgs)
show(MH_poly_Higgs.subs(d==1).simplify_full())

CONJUGACY CLASS 1/3
w = 
[1 0]
[0 1]
Order of centralizer: 6
Number of conjugacy classes: 3
CONTRIBUTION: 1/3*(3*t^2*u^2 - (2*t*u - 1)*(t*u + 1))^d*(3*t^2*v^2 - (2*t*v - 1)*(t*v + 1))^d + 1/6*((t*u + 1)^2)^d*((t*v + 1)^2)^d + 1/2*(-(t*u + 1)*(t*u - 1))^d*(-(t*v + 1)*(t*v - 1))^d
----------------

CONJUGACY CLASS 2/3
w = 
[-1  0]
[ 1  1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (t^2*u*v)^d*(t*u + 1)^d*(t*v + 1)^d
----------------

CONJUGACY CLASS 3/3
w = 
[-1 -1]
[ 1  0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: 3^(2*d)*(t^2*u*v)^(2*d)
----------------

Moduli space of Higgs bundles


(t^2*u*v)^d*(t*u + 1)^d*(t*v + 1)^d + 3^(2*d)*(t^2*u*v)^(2*d) + 1/6*(t^2*u^2 + 2*t*u + 1)^d*(t^2*v^2 + 2*t*v + 1)^d + 1/3*(t^2*u^2 - t*u + 1)^d*(t^2*v^2 - t*v + 1)^d + 1/2*(-t^2*u^2 + 1)^d*(-t^2*v^2 + 1)^d

(11*t^4*u^2 + t^3*u)*v^2 + (t^3*u^2 + 2*t^2*u)*v + 1

In [8]:
# The Poincare polynomials agree, as predicted by the non-abelian Hodge correspondence.

show(MH_poly.subs(r==2*d).subs(q==1).simplify_full())
show(MH_poly_Higgs.subs(u==1).subs(v==1).simplify_full())

3^(2*d)*(t^2)^(2*d) + (t^2)^d*(t + 1)^(2*d) + 1/6*(t^2 + 2*t + 1)^(2*d) + 1/3*(t^2 - t + 1)^(2*d) + 1/2*(-t^2 + 1)^(2*d)

3^(2*d)*(t^2)^(2*d) + (t^2)^d*(t + 1)^(2*d) + 1/6*(t^2 + 2*t + 1)^(2*d) + 1/3*(t^2 - t + 1)^(2*d) + 1/2*(-t^2 + 1)^(2*d)

## Examples for classical groups

Here, we provide examples of calculations for several classical groups, including $\text{GL}_n$, $\text{SL}_n$, $\text{PGL}_n$, $\text{SO}_n$ and $\text{Sp}_{2n}$ for low $n$.

#### $\text{SL}_2$: Dynkin diagram of type $A_1$ and character lattice equal to root lattice

In [9]:
R = RootSystem("A1")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/2
w = 
[1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 1/2*(q*t + 1)^r + 1/2*(-q*t + 1)^r
----------------

CONJUGACY CLASS 2/2
w = 
[-1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 2^r*(q*t^2)^(1/2*r)
----------------



2^r*(q*t^2)^(1/2*r) + 1/2*(q*t + 1)^r + 1/2*(-q*t + 1)^r

q^2*t^2 + 4*q*t^2 + 1

#### $\text{PGL}_2$: Dynkin diagram of type $A_1$ and character lattice equal to weight lattice (Langlands dual to $\text{SL}_2$)

In [10]:
R = RootSystem("A1")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(P.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/2
w = 
[1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 1/2*(q*t + 1)^r + 1/2*(-q*t + 1)^r
----------------

CONJUGACY CLASS 2/2
w = 
[-1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 2^r*(q*t^2)^(1/2*r)
----------------



2^r*(q*t^2)^(1/2*r) + 1/2*(q*t + 1)^r + 1/2*(-q*t + 1)^r

q^2*t^2 + 4*q*t^2 + 1

#### $\text{SL}_3, \text{PGL}_3$: Langlands dual groups of Dynkin diagram of type $A_2$

In [11]:
R = RootSystem("A2")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/3
w = 
[1 0]
[0 1]
Order of centralizer: 6
Number of conjugacy classes: 3
CONTRIBUTION: 1/3*(3*q^2*t^2 - (2*q*t - 1)*(q*t + 1))^r + 1/6*((q*t + 1)^2)^r + 1/2*(-(q*t + 1)*(q*t - 1))^r
----------------

CONJUGACY CLASS 2/3
w = 
[-1  0]
[ 1  1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (q*t^2)^(1/2*r)*(q*t + 1)^r
----------------

CONJUGACY CLASS 3/3
w = 
[-1 -1]
[ 1  0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: 3^r*(q*t^2)^r
----------------



3^r*(q*t^2)^r + (q*t^2)^(1/2*r)*(q*t + 1)^r + 1/6*(q^2*t^2 + 2*q*t + 1)^r + 1/3*(q^2*t^2 - q*t + 1)^r + 1/2*(-q^2*t^2 + 1)^r

q^4*t^4 + q^3*t^4 + 9*q^2*t^4 + 2*q^2*t^3 + q^2*t^2 + q*t^2 + 1

#### $\text{SL}_4, \text{PGL}_4$: Langlands dual groups of Dynkin diagram of type $A_3$

In [12]:
R = RootSystem("A3")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/5
w = 
[1 0 0]
[0 1 0]
[0 0 1]
Order of centralizer: 24
Number of conjugacy classes: 5
CONTRIBUTION: 1/4*(-4*q^3*t^3 + (3*q^2*t^2 - 2*q*t + 1)*(q*t + 1))^r + 1/8*(-8*(q*t - 1)*q^2*t^2 + (3*q*t + 1)*(3*q*t - 1)*(q*t - 1))^r + 1/24*((q*t + 1)^3)^r + 1/4*(-(q*t + 1)^2*(q*t - 1))^r + 1/3*((3*q^2*t^2 - (2*q*t - 1)*(q*t + 1))*(q*t + 1))^r
----------------

CONJUGACY CLASS 2/5
w = 
[-1 -1 -1]
[ 1  1  0]
[ 0 -1  0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: (q*t^2)^r*(q*t + 1)^r
----------------

CONJUGACY CLASS 3/5
w = 
[-1  0  0]
[ 1  1  0]
[ 0  0  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*((8*q^2*t^2 - (3*q*t + 1)*(3*q*t - 1))^r + ((q*t + 1)^2)^r)*(q*t^2)^(1/2*r)
----------------

CONJUGACY CLASS 4/5
w = 
[-1  0  0]
[ 1  1  1]
[ 0  0 -1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 1/2*(q*t^2)^r*((2*q*t + 2)^r + (-2*q*t + 2)^r)
----------------

CONJUGACY CLASS 5/5
w = 
[-1 -1 -1]
[ 1  0  0

4^r*(q*t^2)^(3/2*r) + 1/2*(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/2*(-q^2*t^2 + 1)^r*(q*t^2)^(1/2*r) + 1/2*(q*t^2)^r*((2*q*t + 2)^r + 2*(q*t + 1)^r + (-2*q*t + 2)^r) + 1/24*(q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r + 1/8*(q^3*t^3 - q^2*t^2 - q*t + 1)^r + 1/3*(q^3*t^3 + 1)^r + 1/4*(-q^3*t^3 + q^2*t^2 - q*t + 1)^r + 1/4*(-q^3*t^3 - q^2*t^2 + q*t + 1)^r

q^6*t^6 + q^5*t^6 + 5*q^4*t^6 + 2*q^4*t^5 + 16*q^3*t^6 + q^4*t^4 + 2*q^3*t^5 + 2*q^3*t^4 + 5*q^2*t^4 + 2*q^2*t^3 + q^2*t^2 + q*t^2 + 1

#### $\text{GL}_2$: Dynkin diagram of type $A_1$ and character lattice equal to ambient lattice

In [13]:
R = RootSystem("A1")

ambient = R.ambient_lattice()
L = ambient.submodule(ambient.basis())
W = ambient.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/2
w = 
[1 0]
[0 1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 1/2*(-q^2*t^2 + 1)^r + 1/2*((q*t + 1)^2)^r
----------------

CONJUGACY CLASS 2/2
w = 
[0 1]
[1 0]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (q*t^2)^(1/2*r)*(q*t + 1)^r
----------------



(q*t^2)^(1/2*r)*(q*t + 1)^r + 1/2*(q^2*t^2 + 2*q*t + 1)^r + 1/2*(-q^2*t^2 + 1)^r

q^4*t^4 + q^3*t^4 + 2*q^3*t^3 + 2*q^2*t^3 + 2*q^2*t^2 + q*t^2 + 2*q*t + 1

#### $\text{GL}_3$: Dynkin diagram of type $A_2$ and character lattice equal to ambient lattice

In [14]:
R = RootSystem("A2")

ambient = R.ambient_lattice()
L = ambient.submodule(ambient.basis())
W = ambient.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/3
w = 
[1 0 0]
[0 1 0]
[0 0 1]
Order of centralizer: 6
Number of conjugacy classes: 3
CONTRIBUTION: 1/3*(q^3*t^3 + 1)^r + 1/2*(-(q*t + 1)*q^2*t^2 + q*t + 1)^r + 1/6*((q*t + 1)^3)^r
----------------

CONJUGACY CLASS 2/3
w = 
[0 0 1]
[1 0 0]
[0 1 0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: (q*t^2)^r*(q*t + 1)^r
----------------

CONJUGACY CLASS 3/3
w = 
[0 0 1]
[0 1 0]
[1 0 0]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (q*t^2)^(1/2*r)*((q*t + 1)^2)^r
----------------



(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^(1/2*r) + (q*t^2)^r*(q*t + 1)^r + 1/6*(q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r + 1/3*(q^3*t^3 + 1)^r + 1/2*(-q^3*t^3 - q^2*t^2 + q*t + 1)^r

q^6*t^6 + q^5*t^6 + 2*q^5*t^5 + q^4*t^6 + 4*q^4*t^5 + 2*q^4*t^4 + 2*q^3*t^5 + 6*q^3*t^4 + 2*q^3*t^3 + q^2*t^4 + 4*q^2*t^3 + 2*q^2*t^2 + q*t^2 + 2*q*t + 1

#### $\text{GL}_4$: Dynkin diagram of type $A_3$ and character lattice equal to ambient lattice

In [15]:
R = RootSystem("A3")

ambient = R.ambient_lattice()
L = ambient.submodule(ambient.basis())
W = ambient.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/5
w = 
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Order of centralizer: 24
Number of conjugacy classes: 5
CONTRIBUTION: 1/24*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/3*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/8*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/4*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 1/4*(-q^4*t^4 + 1)^r
----------------

CONJUGACY CLASS 2/5
w = 
[0 0 0 1]
[0 0 1 0]
[1 0 0 0]
[0 1 0 0]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: (q*t^2)^(3/2*r)*(q*t + 1)^r
----------------

CONJUGACY CLASS 3/5
w = 
[0 0 0 1]
[0 1 0 0]
[0 0 1 0]
[1 0 0 0]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*(((q*t + 1)^3)^r + (-(q^2*t^2 - 1)*(q*t + 1))^r)*(q*t^2)^(1/2*r)
----------------

CONJUGACY CLASS 4/5
w = 
[0 0 0 1]
[0 1 0 0]
[1 0 0 0]
[0 0 1 0]
Order of centralizer: 3
Number of conjugacy classes: 3
CONTRIBUTION: (q*t^2)^r*((q*t + 1)^2)^r
----------------

CONJUGACY CLASS 5/5
w = 
[0 0 0 1]
[0 0 1 0]
[0 1 0 0]
[1 0 0 0]
Order of cen

1/2*(q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/2*(-q^3*t^3 - q^2*t^2 + q*t + 1)^r*(q*t^2)^(1/2*r) + 3/2*(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^r + 1/2*(-q^2*t^2 + 1)^r*(q*t^2)^r + (q*t^2)^(3/2*r)*(q*t + 1)^r + 1/24*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/3*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/8*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/4*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 1/4*(-q^4*t^4 + 1)^r

q^8*t^8 + q^7*t^8 + 2*q^7*t^7 + 2*q^6*t^8 + 4*q^6*t^7 + q^5*t^8 + 2*q^6*t^6 + 6*q^5*t^7 + 7*q^5*t^6 + 2*q^4*t^7 + 2*q^5*t^5 + 8*q^4*t^6 + 8*q^4*t^5 + q^3*t^6 + 2*q^4*t^4 + 6*q^3*t^5 + 7*q^3*t^4 + 2*q^3*t^3 + 2*q^2*t^4 + 4*q^2*t^3 + 2*q^2*t^2 + q*t^2 + 2*q*t + 1

#### $\text{GL}_5$: Dynkin diagram of type $A_4$ and character lattice equal to ambient lattice

In [16]:
R = RootSystem("A4")

ambient = R.ambient_lattice()
L = ambient.submodule(ambient.basis())
W = ambient.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/7
w = 
[1 0 0 0 0]
[0 1 0 0 0]
[0 0 1 0 0]
[0 0 0 1 0]
[0 0 0 0 1]
Order of centralizer: 120
Number of conjugacy classes: 7
CONTRIBUTION: 1/120*(q^5*t^5 + 5*q^4*t^4 + 10*q^3*t^3 + 10*q^2*t^2 + 5*q*t + 1)^r + 1/6*(q^5*t^5 + 2*q^4*t^4 + q^3*t^3 + q^2*t^2 + 2*q*t + 1)^r + 1/8*(q^5*t^5 + q^4*t^4 - 2*q^3*t^3 - 2*q^2*t^2 + q*t + 1)^r + 1/5*(q^5*t^5 + 1)^r + 1/4*(-q^5*t^5 - q^4*t^4 + q*t + 1)^r + 1/12*(-q^5*t^5 - 3*q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 + 3*q*t + 1)^r + 1/6*(-q^5*t^5 + q^3*t^3 - q^2*t^2 + 1)^r
----------------

CONJUGACY CLASS 2/7
w = 
[0 0 0 0 1]
[0 0 0 1 0]
[0 0 1 0 0]
[1 0 0 0 0]
[0 1 0 0 0]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: (q*t^2)^(3/2*r)*((q*t + 1)^2)^r
----------------

CONJUGACY CLASS 3/7
w = 
[0 0 0 0 1]
[0 0 0 1 0]
[0 1 0 0 0]
[0 0 1 0 0]
[1 0 0 0 0]
Order of centralizer: 6
Number of conjugacy classes: 6
CONTRIBUTION: (q*t^2)^(3/2*r)*((q*t + 1)^2)^r
----------------

CONJUGACY CLASS 4/7
w = 
[0 0 0 0 1]
[0 0 0 1 0]
[0 1 

2*(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^(3/2*r) + 1/6*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/3*(q^4*t^4 + q^3*t^3 + q*t + 1)^r*(q*t^2)^(1/2*r) + 1/2*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r*(q*t^2)^(1/2*r) + (q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r*(q*t^2)^r + (-q^3*t^3 - q^2*t^2 + q*t + 1)^r*(q*t^2)^r + (q*t^2)^(2*r)*(q*t + 1)^r + 1/120*(q^5*t^5 + 5*q^4*t^4 + 10*q^3*t^3 + 10*q^2*t^2 + 5*q*t + 1)^r + 1/6*(q^5*t^5 + 2*q^4*t^4 + q^3*t^3 + q^2*t^2 + 2*q*t + 1)^r + 1/8*(q^5*t^5 + q^4*t^4 - 2*q^3*t^3 - 2*q^2*t^2 + q*t + 1)^r + 1/5*(q^5*t^5 + 1)^r + 1/4*(-q^5*t^5 - q^4*t^4 + q*t + 1)^r + 1/12*(-q^5*t^5 - 3*q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 + 3*q*t + 1)^r + 1/6*(-q^5*t^5 + q^3*t^3 - q^2*t^2 + 1)^r

q^10*t^10 + q^9*t^10 + 2*q^9*t^9 + 2*q^8*t^10 + 4*q^8*t^9 + 2*q^7*t^10 + 2*q^8*t^8 + 8*q^7*t^9 + q^6*t^10 + 7*q^7*t^8 + 8*q^6*t^9 + 2*q^7*t^7 + 14*q^6*t^8 + 2*q^5*t^9 + 8*q^6*t^7 + 12*q^5*t^8 + 2*q^6*t^6 + 16*q^5*t^7 + q^4*t^8 + 8*q^5*t^6 + 8*q^4*t^7 + 2*q^5*t^5 + 14*q^4*t^6 + 8*q^4*t^5 + 2*q^3*t^6 + 2*q^4*t^4 + 8*q^3*t^5 + 7*q^3*t^4 + 2*q^3*t^3 + 2*q^2*t^4 + 4*q^2*t^3 + 2*q^2*t^2 + q*t^2 + 2*q*t + 1

In [17]:
show(MH_poly.subs(r == 2).simplify_full().expand())

q^10*t^10 + q^9*t^10 + 2*q^9*t^9 + 2*q^8*t^10 + 4*q^8*t^9 + 2*q^7*t^10 + 2*q^8*t^8 + 8*q^7*t^9 + q^6*t^10 + 7*q^7*t^8 + 8*q^6*t^9 + 2*q^7*t^7 + 14*q^6*t^8 + 2*q^5*t^9 + 8*q^6*t^7 + 12*q^5*t^8 + 2*q^6*t^6 + 16*q^5*t^7 + q^4*t^8 + 8*q^5*t^6 + 8*q^4*t^7 + 2*q^5*t^5 + 14*q^4*t^6 + 8*q^4*t^5 + 2*q^3*t^6 + 2*q^4*t^4 + 8*q^3*t^5 + 7*q^3*t^4 + 2*q^3*t^3 + 2*q^2*t^4 + 4*q^2*t^3 + 2*q^2*t^2 + q*t^2 + 2*q*t + 1

#### $\text{SO}_5$: Dynkin diagram of type $B_2$ and character lattice equal to root lattice

Langlands dual to $\text{Sp}_4$ (Dynkin diagram $C_2$ and character lattice equal to weight lattice).

In [18]:
R = RootSystem("B2")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/5
w = 
[1 0]
[0 1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 1/4*(2*q^2*t^2 - (q*t + 1)*(q*t - 1))^r + 1/8*((q*t + 1)^2)^r + 1/2*(-(q*t + 1)*(q*t - 1))^r + 1/8*((q*t - 1)^2)^r
----------------

CONJUGACY CLASS 2/5
w = 
[-1 -1]
[ 2  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 2^r*(q*t^2)^r
----------------

CONJUGACY CLASS 3/5
w = 
[-1  0]
[ 0 -1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 1/2*(4^r + 2^r)*(q*t^2)^r
----------------

CONJUGACY CLASS 4/5
w = 
[-1  0]
[ 2  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*(q*t^2)^(1/2*r)*((q*t + 1)^r + (-q*t + 1)^r)
----------------

CONJUGACY CLASS 5/5
w = 
[-1 -1]
[ 0  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*(q*t^2)^(1/2*r)*((2*q*t + 2)^r + (-2*q*t + 2)^r)
----------------



1/8*(4^(r + 1) + 3*2^(r + 2))*(q*t^2)^r + 1/2*(q*t^2)^(1/2*r)*((2*q*t + 2)^r + (q*t + 1)^r + (-q*t + 1)^r + (-2*q*t + 2)^r) + 1/8*(q^2*t^2 + 2*q*t + 1)^r + 1/8*(q^2*t^2 - 2*q*t + 1)^r + 1/4*(q^2*t^2 + 1)^r + 1/2*(-q^2*t^2 + 1)^r

q^4*t^4 + 5*q^3*t^4 + 14*q^2*t^4 + q^2*t^2 + 5*q*t^2 + 1

#### $\text{SO}_7$: Dynkin diagram of type $B_3$ and character lattice equal to root lattice

Langlands dual to $\text{Sp}_6$ (Dynkin diagram $C_3$ and character lattice equal to weight lattice).

In [19]:
R = RootSystem("B3")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/10
w = 
[1 0 0]
[0 1 0]
[0 0 1]
Order of centralizer: 48
Number of conjugacy classes: 10
	 Computed 10/10
CONTRIBUTION: 1/6*(2*q^3*t^3 - 2*(q*t - 1)*q^2*t^2 - (q^2*t^2 - (2*q*t + 1)*(q*t - 1))*(q*t - 1))^r + 1/6*(-2*q^3*t^3 + (q^2*t^2 - q*t + 1)*(q*t + 1))^r + 1/48*((q*t + 1)^3)^r + 3/16*(-(q*t + 1)^2*(q*t - 1))^r + 3/16*((q*t + 1)*(q*t - 1)^2)^r + 1/48*(-(q*t - 1)^3)^r + 1/8*((2*q^2*t^2 - (q*t + 1)*(q*t - 1))*(q*t + 1))^r + 1/8*(-(2*q^2*t^2 - (q*t + 1)*(q*t - 1))*(q*t - 1))^r
----------------

CONJUGACY CLASS 2/10
w = 
[-1 -2 -1]
[ 0  1  1]
[ 0  0 -1]
Order of centralizer: 16
Number of conjugacy classes: 10
	 Computed 10/10
CONTRIBUTION: 1/4*(q*t^2)^r*((4*q*t + 4)^r + (2*q*t + 2)^r + (-2*q*t + 2)^r + (-4*q*t + 4)^r)
----------------

CONJUGACY CLASS 3/10
w = 
[-1 -2 -1]
[ 1  1  0]
[ 0  0  1]
Order of centralizer: 8
Number of conjugacy classes: 8
CONTRIBUTION: 1/2*(q*t^2)^r*((2*q*t + 2)^r + (-2*q*t + 2)^r)
----------------

CONJUGACY CLASS 4/10
w = 
[-1 -2 -1]
[ 1  1 

1/48*(8^(r + 1) + 18*4^(r + 1) + 2^(r + 6))*(q*t^2)^(3/2*r) + 1/8*(2*q^2*t^2 + 4*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/8*(2*q^2*t^2 - 4*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/4*(2*q^2*t^2 + 2)^r*(q*t^2)^(1/2*r) + 1/4*(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/4*(q^2*t^2 - 2*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/2*(-q^2*t^2 + 1)^r*(q*t^2)^(1/2*r) + 1/2*(-2*q^2*t^2 + 2)^r*(q*t^2)^(1/2*r) + 1/4*(q*t^2)^r*((4*q*t + 4)^r + 5*(2*q*t + 2)^r + 2*(q*t + 1)^r + 2*(-q*t + 1)^r + 5*(-2*q*t + 2)^r + (-4*q*t + 4)^r) + 1/48*(q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r + 1/8*(q^3*t^3 + q^2*t^2 + q*t + 1)^r + 3/16*(q^3*t^3 - q^2*t^2 - q*t + 1)^r + 1/6*(q^3*t^3 + 1)^r + 1/48*(-q^3*t^3 + 3*q^2*t^2 - 3*q*t + 1)^r + 1/8*(-q^3*t^3 + q^2*t^2 - q*t + 1)^r + 3/16*(-q^3*t^3 - q^2*t^2 + q*t + 1)^r + 1/6*(-q^3*t^3 + 1)^r

q^6*t^6 + 5*q^5*t^6 + 19*q^4*t^6 + 40*q^3*t^6 + q^4*t^4 + 6*q^3*t^4 + 19*q^2*t^4 + q^2*t^2 + 5*q*t^2 + 1

#### $\text{SO}_9$: Dynkin diagram of type $B_4$ and character lattice equal to root lattice

Langlands dual to $\text{Sp}_8$ (Dynkin diagram $C_4$ and character lattice equal to weight lattice).

In [20]:
R = RootSystem("B4")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())

W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
show(MH_poly)
show(MH_poly.subs(r == 2).simplify_full().expand())

CONJUGACY CLASS 1/20
w = 
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Order of centralizer: 384
Number of conjugacy classes: 20
	 Computed 10/20
	 Computed 20/20
CONTRIBUTION: 1/384*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/32*(q^4*t^4 + 2*q^3*t^3 + 2*q^2*t^2 + 2*q*t + 1)^r + 1/12*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/12*(q^4*t^4 - q^3*t^3 - q*t + 1)^r + 1/32*(q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 - 2*q*t + 1)^r + 1/384*(q^4*t^4 - 4*q^3*t^3 + 6*q^2*t^2 - 4*q*t + 1)^r + 1/32*(q^4*t^4 + 2*q^2*t^2 + 1)^r + 7/64*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/8*(q^4*t^4 + 1)^r + 1/24*(-q^4*t^4 + 2*q^3*t^3 - 2*q*t + 1)^r + 1/12*(-q^4*t^4 + q^3*t^3 - q*t + 1)^r + 1/12*(-q^4*t^4 - q^3*t^3 + q*t + 1)^r + 1/24*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 1/4*(-q^4*t^4 + 1)^r
----------------

CONJUGACY CLASS 2/20
w = 
[-1 -2 -2 -1]
[ 0  1  0  0]
[ 0  0  1  1]
[ 0  0  0 -1]
Order of centralizer: 64
Number of conjugacy classes: 25
	 Computed 10/25
	 Computed 20/25
CONTRIBUTION: 1/16*(2*(8*q^2*t^2 - 4*(q*t + 1)*(q*t - 1)

1/384*(16^(r + 1) + 36*8^(r + 1) + 59*4^(r + 2) + 21*2^(r + 5))*(q*t^2)^(2*r) + 1/48*(2*q^3*t^3 + 6*q^2*t^2 + 6*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/8*(2*q^3*t^3 + 2*q^2*t^2 + 2*q*t + 2)^r*(q*t^2)^(1/2*r) + 3/16*(2*q^3*t^3 - 2*q^2*t^2 - 2*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/6*(2*q^3*t^3 + 2)^r*(q*t^2)^(1/2*r) + 1/16*(q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/8*(q^3*t^3 + q^2*t^2 + q*t + 1)^r*(q*t^2)^(1/2*r) + 5/16*(q^3*t^3 - q^2*t^2 - q*t + 1)^r*(q*t^2)^(1/2*r) + 1/16*(-q^3*t^3 + 3*q^2*t^2 - 3*q*t + 1)^r*(q*t^2)^(1/2*r) + 1/8*(-q^3*t^3 + q^2*t^2 - q*t + 1)^r*(q*t^2)^(1/2*r) + 5/16*(-q^3*t^3 - q^2*t^2 + q*t + 1)^r*(q*t^2)^(1/2*r) + 1/48*(-2*q^3*t^3 + 6*q^2*t^2 - 6*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/8*(-2*q^3*t^3 + 2*q^2*t^2 - 2*q*t + 2)^r*(q*t^2)^(1/2*r) + 3/16*(-2*q^3*t^3 - 2*q^2*t^2 + 2*q*t + 2)^r*(q*t^2)^(1/2*r) + 1/6*(-2*q^3*t^3 + 2)^r*(q*t^2)^(1/2*r) + 1/16*(4*q^2*t^2 + 8*q*t + 4)^r*(q*t^2)^r + 1/16*(4*q^2*t^2 - 8*q*t + 4)^r*(q*t^2)^r + 1/8*(4*q^2*t^2 + 4)^r*(q*t^2)^r + 7/16*(2*q^2*t^2 + 4*q*t + 2)^r*(q*t^2)^r + 7/16*(2*q^2*t^2 - 4*q*t + 2)^r*(q*t^2)^r + 3/8*(2*q^2*t^2 + 2)^r*(q*t^2)^r + 3/8*(q^2*t^2 + 2*q*t + 1)^r*(q*t^2)^r + 3/8*(q^2*t^2 - 2*q*t + 1)^r*(q*t^2)^r + 1/4*(q^2*t^2 + 1)^r*(q*t^2)^r + (-q^2*t^2 + 1)^r*(q*t^2)^r + 5/4*(-2*q^2*t^2 + 2)^r*(q*t^2)^r + 1/4*(-4*q^2*t^2 + 4)^r*(q*t^2)^r + 1/12*(q*t^2)^(3/2*r)*((8*q*t + 8)^r + 12*(4*q*t + 4)^r + 23*(2*q*t + 2)^r + 6*(q*t + 1)^r + 6*(-q*t + 1)^r + 23*(-2*q*t + 2)^r + 12*(-4*q*t + 4)^r + (-8*q*t + 8)^r) + 1/384*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/32*(q^4*t^4 + 2*q^3*t^3 + 2*q^2*t^2 + 2*q*t + 1)^r + 1/12*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/12*(q^4*t^4 - q^3*t^3 - q*t + 1)^r + 1/32*(q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 - 2*q*t + 1)^r + 1/384*(q^4*t^4 - 4*q^3*t^3 + 6*q^2*t^2 - 4*q*t + 1)^r + 1/32*(q^4*t^4 + 2*q^2*t^2 + 1)^r + 7/64*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/8*(q^4*t^4 + 1)^r + 1/24*(-q^4*t^4 + 2*q^3*t^3 - 2*q*t + 1)^r + 1/12*(-q^4*t^4 + q^3*t^3 - q*t + 1)^r + 1/12*(-q^4*t^4 - q^3*t^3 + q*t + 1)^r + 1/24*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 1/4*(-q^4*t^4 + 1)^r

q^8*t^8 + 5*q^7*t^8 + 20*q^6*t^8 + 59*q^5*t^8 + q^6*t^6 + 105*q^4*t^8 + 6*q^5*t^6 + 25*q^4*t^6 + 59*q^3*t^6 + q^4*t^4 + 6*q^3*t^4 + 20*q^2*t^4 + q^2*t^2 + 5*q*t^2 + 1

#### $\text{SO}_8$: Dynkin diagram of type $D_4$ and character lattice intermediate between the root and the weight lattice

In [21]:
R = RootSystem("D4")

Q = R.root_lattice()
P = R.weight_lattice()
omega = P.fundamental_weights()

SO8_basis = [omega[1], omega[2], omega[3] + omega[4], omega[3] - omega[4]]

L = P.submodule(SO8_basis)
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/13
w = 
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Order of centralizer: 192
Number of conjugacy classes: 13
	 Computed 10/13
CONTRIBUTION: 1/192*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/6*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/6*(q^4*t^4 - q^3*t^3 - q*t + 1)^r + 1/192*(q^4*t^4 - 4*q^3*t^3 + 6*q^2*t^2 - 4*q*t + 1)^r + 1/16*(q^4*t^4 + 2*q^2*t^2 + 1)^r + 3/32*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/16*(-q^4*t^4 + 2*q^3*t^3 - 2*q*t + 1)^r + 1/16*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 3/8*(-q^4*t^4 + 1)^r
----------------

CONJUGACY CLASS 2/13
w = 
[-1 -2 -1 -1]
[ 0  1  0  1]
[ 0  0  0 -1]
[ 0  0  1  0]
Order of centralizer: 8
Number of conjugacy classes: 8
CONTRIBUTION: 1/2*(q*t^2)^(3/2*r)*((4*q*t + 4)^r + (-4*q*t + 4)^r)
----------------

CONJUGACY CLASS 3/13
w = 
[-1 -2 -1 -1]
[ 1  1  0  1]
[ 0  0  0 -1]
[ 0  0  1  0]
Order of centralizer: 16
Number of conjugacy classes: 10
	 Computed 10/10
CONTRIBUTION: 1/2*(4^r + 2^r)*(q*t^2)^(2*r)
----------------

CONJUGACY CLASS 4/13

#### $\text{SO}_{10}$: Dynkin diagram of type $D_5$ and character lattice intermediate between the root and the weight lattice

In [22]:
R = RootSystem("D5")

Q = R.root_lattice()
P = R.weight_lattice()
omega = P.fundamental_weights()

SO10_basis = [omega[1], omega[2], omega[3], omega[4] + omega[5], omega[4] - omega[5]]

L = P.submodule(SO10_basis)
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/18
w = 
[1 0 0 0 0]
[0 1 0 0 0]
[0 0 1 0 0]
[0 0 0 1 0]
[0 0 0 0 1]
Order of centralizer: 1920
Number of conjugacy classes: 18
	 Computed 10/18
CONTRIBUTION: 1/1920*(q^5*t^5 + 5*q^4*t^4 + 10*q^3*t^3 + 10*q^2*t^2 + 5*q*t + 1)^r + 1/24*(q^5*t^5 + 2*q^4*t^4 + q^3*t^3 + q^2*t^2 + 2*q*t + 1)^r + 1/32*(q^5*t^5 + q^4*t^4 + 2*q^3*t^3 + 2*q^2*t^2 + q*t + 1)^r + 7/192*(q^5*t^5 + q^4*t^4 - 2*q^3*t^3 - 2*q^2*t^2 + q*t + 1)^r + 1/16*(q^5*t^5 - q^4*t^4 - q*t + 1)^r + 1/24*(q^5*t^5 - 2*q^4*t^4 + q^3*t^3 + q^2*t^2 - 2*q*t + 1)^r + 1/384*(q^5*t^5 - 3*q^4*t^4 + 2*q^3*t^3 + 2*q^2*t^2 - 3*q*t + 1)^r + 1/12*(q^5*t^5 - q^3*t^3 - q^2*t^2 + 1)^r + 1/5*(q^5*t^5 + 1)^r + 1/96*(-q^5*t^5 + 3*q^4*t^4 - 4*q^3*t^3 + 4*q^2*t^2 - 3*q*t + 1)^r + 1/32*(-q^5*t^5 + q^4*t^4 + 2*q^3*t^3 - 2*q^2*t^2 - q*t + 1)^r + 1/8*(-q^5*t^5 + q^4*t^4 - q*t + 1)^r + 5/32*(-q^5*t^5 - q^4*t^4 + q*t + 1)^r + 1/96*(-q^5*t^5 - 3*q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 + 3*q*t + 1)^r + 1/12*(-q^5*t^5 + q^3*t^3 - q^2*t^2 + 1)^r + 1/12*

## Semisimple exceptional groups

#### $\text{G}_{2}$

In [23]:
R = RootSystem("G2")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/6
w = 
[1 0]
[0 1]
Order of centralizer: 12
Number of conjugacy classes: 6
CONTRIBUTION: 1/4*(3*q^2*t^2 - (2*q*t + 1)*(2*q*t - 1))^r + 1/6*(3*q^2*t^2 - (2*q*t - 1)*(q*t + 1))^r + 1/6*(3*q^2*t^2 - (2*q*t + 1)*(q*t - 1))^r + 1/12*((q*t + 1)^2)^r + 1/4*(-(q*t + 1)*(q*t - 1))^r + 1/12*((q*t - 1)^2)^r
----------------

CONJUGACY CLASS 2/6
w = 
[-2 -3]
[ 1  2]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*(q*t^2)^(1/2*r)*((q*t + 1)^r + (-q*t + 1)^r)
----------------

CONJUGACY CLASS 3/6
w = 
[-1 -3]
[ 0  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*(q*t^2)^(1/2*r)*((q*t + 1)^r + (-q*t + 1)^r)
----------------

CONJUGACY CLASS 4/6
w = 
[-1 -3]
[ 1  2]
Order of centralizer: 6
Number of conjugacy classes: 6
CONTRIBUTION: (q*t^2)^r
----------------

CONJUGACY CLASS 5/6
w = 
[-1  0]
[ 0 -1]
Order of centralizer: 12
Number of conjugacy classes: 6
CONTRIBUTION: 1/6*(4^r + 3*2^r + 2)*(q*t^2)^r
----------------

CONJUGACY CLA

#### $\text{F}_{4}$

In [24]:
R = RootSystem("F4")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/25
w = 
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 1]
Order of centralizer: 1152
Number of conjugacy classes: 25
	 Computed 10/25
	 Computed 20/25
CONTRIBUTION: 1/1152*(q^4*t^4 + 4*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/72*(q^4*t^4 + 2*q^3*t^3 + 3*q^2*t^2 + 2*q*t + 1)^r + 1/32*(q^4*t^4 + 2*q^3*t^3 + 2*q^2*t^2 + 2*q*t + 1)^r + 1/18*(q^4*t^4 + q^3*t^3 + q*t + 1)^r + 1/18*(q^4*t^4 - q^3*t^3 - q*t + 1)^r + 1/72*(q^4*t^4 - 2*q^3*t^3 + 3*q^2*t^2 - 2*q*t + 1)^r + 1/32*(q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 - 2*q*t + 1)^r + 1/1152*(q^4*t^4 - 4*q^3*t^3 + 6*q^2*t^2 - 4*q*t + 1)^r + 1/96*(q^4*t^4 + 2*q^2*t^2 + 1)^r + 1/12*(q^4*t^4 - q^2*t^2 + 1)^r + 5/64*(q^4*t^4 - 2*q^2*t^2 + 1)^r + 1/8*(q^4*t^4 + 1)^r + 1/48*(-q^4*t^4 + 2*q^3*t^3 - 2*q*t + 1)^r + 1/6*(-q^4*t^4 + q^3*t^3 - q*t + 1)^r + 1/6*(-q^4*t^4 - q^3*t^3 + q*t + 1)^r + 1/48*(-q^4*t^4 - 2*q^3*t^3 + 2*q*t + 1)^r + 1/8*(-q^4*t^4 + 1)^r
----------------

CONJUGACY CLASS 2/25
w = 
[-1  0  0  0]
[ 0 -1  0  0]
[ 0  0 -1  0]
[ 0  0  0 -1]
Ord

#### $\text{E}_{6}$

In [25]:
R = RootSystem("E6")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/25
w = 
[1 0 0 0 0 0]
[0 1 0 0 0 0]
[0 0 1 0 0 0]
[0 0 0 1 0 0]
[0 0 0 0 1 0]
[0 0 0 0 0 1]
Order of centralizer: 51840
Number of conjugacy classes: 25
	 Computed 10/25
	 Computed 20/25
CONTRIBUTION: 1/51840*(q^6*t^6 + 6*q^5*t^5 + 15*q^4*t^4 + 20*q^3*t^3 + 15*q^2*t^2 + 6*q*t + 1)^r + 1/216*(q^6*t^6 + 3*q^5*t^5 + 3*q^4*t^4 + 2*q^3*t^3 + 3*q^2*t^2 + 3*q*t + 1)^r + 1/96*(q^6*t^6 + 2*q^5*t^5 + 3*q^4*t^4 + 4*q^3*t^3 + 3*q^2*t^2 + 2*q*t + 1)^r + 1/192*(q^6*t^6 + 2*q^5*t^5 - q^4*t^4 - 4*q^3*t^3 - q^2*t^2 + 2*q*t + 1)^r + 1/72*(q^6*t^6 + q^5*t^5 + 2*q^4*t^4 + q^3*t^3 + 2*q^2*t^2 + q*t + 1)^r + 1/36*(q^6*t^6 + q^5*t^5 - q^4*t^4 - 2*q^3*t^3 - q^2*t^2 + q*t + 1)^r + 1/10*(q^6*t^6 + q^5*t^5 + q*t + 1)^r + 1/24*(q^6*t^6 - q^5*t^5 - q^4*t^4 + 2*q^3*t^3 - q^2*t^2 - q*t + 1)^r + 1/12*(q^6*t^6 - q^5*t^5 + q^3*t^3 - q*t + 1)^r + 1/36*(q^6*t^6 - 2*q^5*t^5 + 2*q^4*t^4 - 2*q^3*t^3 + 2*q^2*t^2 - 2*q*t + 1)^r + 1/1152*(q^6*t^6 - 2*q^5*t^5 - q^4*t^4 + 4*q^3*t^3 - q^2*t^2 - 2*q*t + 1)^r + 1/6

#### $\text{E}_{7}$

In [26]:
R = RootSystem("E7")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/60
w = 
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]
Order of centralizer: 2903040
Number of conjugacy classes: 60
	 Computed 10/60
	 Computed 20/60
	 Computed 30/60
	 Computed 40/60
	 Computed 50/60
	 Computed 60/60
CONTRIBUTION: 1/2903040*(q^7*t^7 + 7*q^6*t^6 + 21*q^5*t^5 + 35*q^4*t^4 + 35*q^3*t^3 + 21*q^2*t^2 + 7*q*t + 1)^r + 1/4320*(q^7*t^7 + 4*q^6*t^6 + 6*q^5*t^5 + 5*q^4*t^4 + 5*q^3*t^3 + 6*q^2*t^2 + 4*q*t + 1)^r + 1/768*(q^7*t^7 + 3*q^6*t^6 + 5*q^5*t^5 + 7*q^4*t^4 + 7*q^3*t^3 + 5*q^2*t^2 + 3*q*t + 1)^r + 1/3072*(q^7*t^7 + 3*q^6*t^6 + q^5*t^5 - 5*q^4*t^4 - 5*q^3*t^3 + q^2*t^2 + 3*q*t + 1)^r + 1/144*(q^7*t^7 + 2*q^6*t^6 + 3*q^5*t^5 + 3*q^4*t^4 + 3*q^3*t^3 + 3*q^2*t^2 + 2*q*t + 1)^r + 1/60*(q^7*t^7 + 2*q^6*t^6 + q^5*t^5 + q^2*t^2 + 2*q*t + 1)^r + 1/288*(q^7*t^7 + 2*q^6*t^6 - 3*q^4*t^4 - 3*q^3*t^3 + 2*q*t + 1)^r + 1/32*(q^7*t^7 + q^6*t^6 + q^5*t^5 + q^4*t^4 + q^3*t^3 + q^2*t^2 + q*t + 1)^r + 7/384*(q

#### $\text{E}_{8}$

In [27]:
R = RootSystem("E8")

Q = R.root_lattice()
P = R.weight_lattice()

L = P.submodule(Q.basis())
W = P.weyl_group()

MH_poly = compute_stringy_MHpolynomial(L, W,  r0 = r, compact_support = False, Higgs = False)
print(MH_poly)

CONJUGACY CLASS 1/112
w = 
[1 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0]
[0 0 0 1 0 0 0 0]
[0 0 0 0 1 0 0 0]
[0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 1]
Order of centralizer: 696729600
Number of conjugacy classes: 112
	 Computed 10/112
	 Computed 20/112
	 Computed 30/112
	 Computed 40/112
	 Computed 50/112
	 Computed 60/112
	 Computed 70/112
	 Computed 80/112
	 Computed 90/112
	 Computed 100/112
	 Computed 110/112
CONTRIBUTION: 1/696729600*(q^8*t^8 + 8*q^7*t^7 + 28*q^6*t^6 + 56*q^5*t^5 + 70*q^4*t^4 + 56*q^3*t^3 + 28*q^2*t^2 + 8*q*t + 1)^r + 1/311040*(q^8*t^8 + 5*q^7*t^7 + 10*q^6*t^6 + 11*q^5*t^5 + 10*q^4*t^4 + 11*q^3*t^3 + 10*q^2*t^2 + 5*q*t + 1)^r + 1/155520*(q^8*t^8 + 4*q^7*t^7 + 10*q^6*t^6 + 16*q^5*t^5 + 19*q^4*t^4 + 16*q^3*t^3 + 10*q^2*t^2 + 4*q*t + 1)^r + 1/18432*(q^8*t^8 + 4*q^7*t^7 + 8*q^6*t^6 + 12*q^5*t^5 + 14*q^4*t^4 + 12*q^3*t^3 + 8*q^2*t^2 + 4*q*t + 1)^r + 1/184320*(q^8*t^8 + 4*q^7*t^7 + 4*q^6*t^6 - 4*q^5*t^5 - 10*q^4*t^4 - 4*q^3*t^3 + 4*q^2*t^2 + 4*q*t + 1

## Examples of Langlands duality

Here, we empirically check Langlands duality for some families of Langlands dual groups.

#### $G = \text{SL}_n$ and $^LG = \text{PGL}_n$ for $2 \leq n \leq 10$

Both groups are of Dynkin type $A_n$, and the character lattice of $\text{SL}_n$ is the root lattice ($\Lambda = Q$), whereas the character lattice of $\text{PGL}_n$ is the weight lattice ($\Lambda = P$).

In [28]:
results_G = []
results_Gdual = []

for rk in range(1,10):
    R = RootSystem("A" + str(rk))

    Q = R.root_lattice()
    P = R.weight_lattice()

    L = P.submodule(Q.basis())
    L_dual = P.submodule(P.basis())
    W = P.weyl_group()

    MH_poly_G = compute_stringy_MHpolynomial(L, W,  r0 = 2, compact_support = False, Higgs = False)
    MH_poly_Gdual = compute_stringy_MHpolynomial(L_dual, W,  r0 = 2, compact_support = False, Higgs = False)

    results_G.append(MH_poly_G)
    results_Gdual.append(MH_poly_Gdual)

for i in range(len(results_G)):
    print('Difference for SL_' + str(i + 2) + ' and PGL_' + str(i + 2) + ': ' + str((MH_poly_G-MH_poly_Gdual).simplify_full()))

CONJUGACY CLASS 1/2
w = 
[1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 1/2*(q*t + 1)^2 + 1/2*(q*t - 1)^2
----------------

CONJUGACY CLASS 2/2
w = 
[-1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 4*q*t^2
----------------

CONJUGACY CLASS 1/2
w = 
[1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 1/2*(q*t + 1)^2 + 1/2*(q*t - 1)^2
----------------

CONJUGACY CLASS 2/2
w = 
[-1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: 4*q*t^2
----------------

CONJUGACY CLASS 1/3
w = 
[1 0]
[0 1]
Order of centralizer: 6
Number of conjugacy classes: 3
CONTRIBUTION: 1/6*(q*t + 1)^4 + 1/2*(q*t + 1)^2*(q*t - 1)^2 + 1/3*(3*q^2*t^2 - (2*q*t - 1)*(q*t + 1))^2
----------------

CONJUGACY CLASS 2/3
w = 
[-1  0]
[ 1  1]
Order of centralizer: 2
Number of conjugacy classes: 2
CONTRIBUTION: (q*t + 1)^2*q*t^2
----------------

CONJUGACY CLASS 3/3
w = 
[-1 -1]
[ 1  0]
Order of centralizer: 3
Number of conjugacy class

#### $G = \text{SO}_{2n+1}$ and $^LG = \text{Sp}_{2n}$ for $2 \leq n \leq 7$

$\text{SO}_{2n+1}$ if of Dynkin type $B_n$ with character lattice equal to the root lattice ($\Lambda = Q$), while $\text{Sp}_{2n}$ is of Dynkin type $C_n$ and the character lattice is the weight lattice ($\Lambda = P$).

In [29]:
results_G = []
results_Gdual = []

for rk in range(2,8):
    R = RootSystem("B" + str(rk))
    R_dual = RootSystem("C" + str(rk))

    Q = R.root_lattice()
    P = R.weight_lattice()
    L = P.submodule(Q.basis())
    W = P.weyl_group()

    P_dual = R_dual.weight_lattice()
    L_dual = P_dual.submodule(P_dual.basis())
    W_dual = P_dual.weyl_group()

    MH_poly_G = compute_stringy_MHpolynomial(L, W,  r0 = 2, compact_support = False, Higgs = False)
    MH_poly_Gdual = compute_stringy_MHpolynomial(L_dual, W_dual,  r0 = 2, compact_support = False, Higgs = False)

    results_G.append(MH_poly_G)
    results_Gdual.append(MH_poly_Gdual)

for i in range(len(results_G)):
    print('Difference for SO_' + str(2*(i + 2)+1) + ' and Sp_' + str(2*i + 2) + ': ' + str((MH_poly_G-MH_poly_Gdual).simplify_full()))

CONJUGACY CLASS 1/5
w = 
[1 0]
[0 1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 1/8*(q*t + 1)^4 + 1/2*(q*t + 1)^2*(q*t - 1)^2 + 1/8*(q*t - 1)^4 + 1/4*(2*q^2*t^2 - (q*t + 1)*(q*t - 1))^2
----------------

CONJUGACY CLASS 2/5
w = 
[-1 -1]
[ 2  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 4*q^2*t^4
----------------

CONJUGACY CLASS 3/5
w = 
[-1  0]
[ 0 -1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 10*q^2*t^4
----------------

CONJUGACY CLASS 4/5
w = 
[-1  0]
[ 2  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 1/2*((q*t + 1)^2 + (q*t - 1)^2)*q*t^2
----------------

CONJUGACY CLASS 5/5
w = 
[-1 -1]
[ 0  1]
Order of centralizer: 4
Number of conjugacy classes: 4
CONTRIBUTION: 2*((q*t + 1)^2 + (q*t - 1)^2)*q*t^2
----------------

CONJUGACY CLASS 1/5
w = 
[1 0]
[0 1]
Order of centralizer: 8
Number of conjugacy classes: 5
CONTRIBUTION: 1/8*(q*t + 1)^4 + 1/2*(q*t + 1)^2*(q*t - 1)^2 + 1/8*(q*t 